# Pipeline d'Inférence d'Images avec TensorFlow
Ce notebook présente un pipeline complet d'inférence d'images à l'aide de TensorFlow et Keras. Il inclut la classification, le débruitage et la génération de légendes.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow import keras
import matplotlib.pyplot as plt
import numpy as np
import os
import json
import pickle
from PIL import Image
from typing import List, Tuple, Dict, Optional, Union

## Paramètres globaux

In [ ]:
IMG_SIZE_CAT = (180, 180)
IMG_SIZE_AUTO_ENCODER = (256, 256)
BATCH_SIZE = 32
DATA_DIR = "./data_validation2"
CLASS_NAMES = ['Painting', 'Photo', 'Schematics', 'Sketch', 'Text']

## Chargement des modèles entraînés

### Définition des caractéristiques des modèles (loss, architecture)

In [ ]:
class CNN_Encoder(tf.keras.Model):
    # Comme les images sont déjà prétraitées par InceptionV3 et représentées sous forme compacte
    # L'encodeur CNN ne fera que transmettre ces caractéristiques à une couche dense
    def __init__(self, embedding_dim, **kwargs):
        kwargs.setdefault('name', 'CNN_Encoder')
        super(CNN_Encoder, self).__init__(**kwargs)
        # forme après fc == (batch_size, 64, embedding_dim)
        self.embedding_dim = embedding_dim
        self.fc = tf.keras.layers.Dense(embedding_dim, activation='relu')

    def call(self, x):
        x = self.fc(x)
        x = tf.nn.relu(x)
        return x

    def get_config(self):
        config = super().get_config()
        config.update({
            "embedding_dim": self.embedding_dim,
        })
        return config

    @classmethod
    def from_config(cls, config):
        name = config.pop('name', None)
        return cls(**config)

In [ ]:
class BahdanauAttention(tf.keras.Model):
    def __init__(self, units):
        super(BahdanauAttention, self).__init__()
        self.W1 = tf.keras.layers.Dense(units)
        self.W2 = tf.keras.layers.Dense(units)
        self.V = tf.keras.layers.Dense(1)
        self.units = units

    def call(self, features, hidden):
        # features(CNN_Encoder output) shape == (batch_size, 64, embedding_dim)

        # shape of hidden == (batch_size, hidden_size)
        # Adding time axis to hidden state to match the shape of features
        hidden_with_time_axis = tf.expand_dims(hidden, 1)

        # Passing through dense layers and adding them
        attention_hidden_layer = tf.nn.tanh(self.W1(features) + self.W2(hidden_with_time_axis))

        # Calculating the attention scores
        score = self.V(attention_hidden_layer)

        # Applying softmax to normalize the scores
        attention_weights = tf.nn.softmax(score, axis=1)

        # Calculating the context vector as the weighted sum of features
        context_vector = attention_weights * features
        context_vector = tf.reduce_sum(context_vector, axis=1)

        return context_vector, attention_weights

In [ ]:
class RNN_Decoder(tf.keras.Model):
    def __init__(self, embedding_dim, units, vocab_size, **kwargs):
        kwargs.setdefault('name', 'rnn_encoder')
        super(RNN_Decoder, self).__init__(**kwargs)
        self.units = units
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        # Embedding layer to convert word indices to dense vectors
        self.embedding = tf.keras.layers.Embedding(vocab_size, embedding_dim)

        # GRU layer
        self.gru = tf.keras.layers.GRU(self.units,
                                       return_sequences=True,
                                       return_state=True,
                                       recurrent_initializer='glorot_uniform')

        # Fully connected layer after GRU
        self.fc1 = tf.keras.layers.Dense(self.units)

        # Final fully connected layer to generate predictions
        self.fc2 = tf.keras.layers.Dense(vocab_size)

        # Attention mechanism
        self.attention = BahdanauAttention(self.units)

    def call(self, x, features, hidden):
        # Attention mechanism
        context_vector, attention_weights = self.attention(features, hidden)

        # Pass the input word through the embedding layer
        x = self.embedding(x)

        # Concatenate the context vector and the embedding
        x = tf.concat([tf.expand_dims(context_vector, 1), x], axis=-1)

        # Pass the concatenated vector through the GRU
        output, state = self.gru(x)

        # Pass the GRU output through the first fully connected layer
        y = self.fc1(output)

        # Reshape the output to prepare for the final fully connected layer
        y = tf.reshape(y, (-1, y.shape[2]))

        # Pass the reshaped output through the final fully connected layer
        y = self.fc2(y)

        return y, state, attention_weights

    def reset_state(self, batch_size):
        return tf.zeros((batch_size, self.units))

    def get_config(self):
        config = super(RNN_Decoder, self).get_config()
        config.update({
            "embedding_dim": self.embedding_dim,
            "units": self.units,
            "vocab_size": self.vocab_size,
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(
            embedding_dim=config["embedding_dim"],
            units=config["units"],
            vocab_size=config["vocab_size"]
        )

In [ ]:
class ImageCaptioningModel:
    """
    Classe pour charger et utiliser un modèle de génération de légendes d'images entraîné
    """
    def __init__(self, model_dir: str = "./saved_models"):
        """
        Initialise le modèle en chargeant les composants nécessaires
        
        Args:
            model_dir: Répertoire contenant les modèles sauvegardés
        """
        self.model_dir = model_dir
        self.encoder = None
        self.decoder = None
        self.feature_extractor = None
        self.tokenizer = None
        self.metadata = None
        self.max_length = None
        self.img_size = None
        image_model = tf.keras.applications.InceptionV3(include_top=False, weights='imagenet')
        self.feature_extractor = tf.keras.Model(
            inputs=image_model.input, 
            outputs=image_model.layers[-1].output
        )
        
        # Charger les composants du modèle
        self._load_models()
    
    def _load_models(self) -> None:
        """Charge les modèles et métadonnées nécessaires"""
        print("Chargement des modèles et métadonnées...")
        
        # Charger les métadonnées
        metadata_path = os.path.join(self.model_dir, 'model_metadata.json')
        with open(metadata_path, 'r') as f:
            self.metadata = json.load(f)
        
        # Extraire les informations importantes
        self.max_length = self.metadata['max_length']
        self.img_size = tuple(self.metadata['img_size'])
        
        # Charger le tokenizer
        tokenizer_path = os.path.join(self.model_dir, 'tokenizer.pickle')
        with open(tokenizer_path, 'rb') as handle:
            self.tokenizer = pickle.load(handle)
        
        # Charger le modèle encodeur (utilisez best_encoder s'il existe, sinon final_encoder)
        encoder_path = os.path.join(self.model_dir, 'best_encoder.keras')
        if not os.path.exists(encoder_path):
            encoder_path = os.path.join(self.model_dir, 'encoder.keras')
        self.encoder = tf.keras.models.load_model(encoder_path, 
                                                  custom_objects={'CNN_Encoder': CNN_Encoder},
                                                  )
        print(self.encoder.summary())
        
        # Charger le modèle décodeur
        decoder_path = os.path.join(self.model_dir, 'best_decoder.keras')
        if not os.path.exists(decoder_path):
            decoder_path = os.path.join(self.model_dir, 'decoder.keras')
        self.decoder = tf.keras.models.load_model(decoder_path, 
                                                  custom_objects={
                                                    'RNN_Decoder': RNN_Decoder                                                },
                                                )
        
        print("Modèles chargés avec succès!")
    
    def _preprocess_image(self, image) -> tf.Tensor:
        """
        Charge et prétraite une image pour l'entrée du modèle
        
        Args:
            image_path: Chemin vers l'image ou array numpy (déjà chargée)
            
        Returns:
            Tenseur de l'image prétraitée
        """
        img = tf.convert_to_tensor(image)
        
        # Redimensionner et prétraiter
        img = tf.image.resize(img, self.img_size)
        img = tf.keras.applications.inception_v3.preprocess_input(img)
        return img
    
    def extract_features(self, image) -> tf.Tensor:
        """
        Extrait les caractéristiques d'une image
        
        Args:
            image_path: Chemin vers l'image ou array numpy (déjà chargée)
            
        Returns:
            Caractéristiques extraites de l'image
        """
        # Prétraiter l'image
        img = self._preprocess_image(image)
        img_tensor = tf.expand_dims(img, 0)
        
        # Extraire les caractéristiques
        features = self.feature_extractor(img_tensor)
        features = tf.reshape(
            features,
            (features.shape[0], -1, features.shape[3])
        )
        
        return features
    
    def generate_caption(
        self, 
        image,
        batch_size: int = 32,
        max_length: int = 50,
    ) -> Union[str, Tuple[str, np.ndarray]]:
        """
        Génère une légende pour une image
        
        Args:
            image_path: Chemin vers l'image ou array numpy (déjà chargée)
            beam_size: Taille du beam search (1 = greedy search)
            temperature: Contrôle la diversité des prédictions
            return_attention: Si True, retourne également les poids d'attention
            
        Returns:
            Légende générée (et matrice d'attention si return_attention=True)
        """
        # Extraire et encoder les caractéristiques
        features = self.extract_features(image)
        encoded_features = self.encoder(features, training=False)
        
        hidden = self.decoder.reset_state(batch_size)
        dec_input = tf.expand_dims([self.tokenizer.word_index['<start>']], 0)

        result = []
        for i in range(max_length):
            predictions, hidden, attention_weights = self.decoder(dec_input, features, hidden)
            
            predicted_id = tf.random.categorical(predictions, 1)[0][0].numpy()
            predicted_word = self.tokenizer.index_word.get(predicted_id, '<unk>')  # Use .get() to handle missing keys
            result.append(predicted_word)

            if predicted_word == '<end>':
                return result
        return result


In [ ]:
@tf.keras.utils.register_keras_serializable()
def weighted_loss(y_true, y_pred):
    weights = tf.gather(class_weight_tensor, tf.cast(y_true, tf.int32))
    unweighted_loss = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return unweighted_loss * weights

In [ ]:
class Autoencoder(tf.keras.Model):
    def __init__(self, encoder, decoder, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        self.latent_dim = latent_dim

    def call(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

    def get_config(self):
        config = super().get_config()
        config.update({
            "encoder": tf.keras.utils.serialize_keras_object(self.encoder),
            "decoder": tf.keras.utils.serialize_keras_object(self.decoder),
            "latent_dim": self.latent_dim
        })
        return config

    @classmethod
    def from_config(cls, config):
        encoder = tf.keras.utils.deserialize_keras_object(config.pop("encoder"))
        decoder = tf.keras.utils.deserialize_keras_object(config.pop("decoder"))
        return cls(encoder=encoder, decoder=decoder, **config)

### Chargement des modèles (Livrable 1 et 2)

In [ ]:
print('Chargement du modèle de classification...')
classification_model = load_model("./L1_model.keras")
print('Chargement du modèle autoencodeur...')
autoencoder = load_model("./L2_model_mse.keras", custom_objects={"Autoencoder": Autoencoder})
print('Chargement du modèle de captionning...')
captionning_model = ImageCaptioningModel(model_dir="./saved_models")
print('Modèles chargés avec succès!')

## Construction du pipeline

### 0 Création du dataset

In [ ]:
raw_ds = keras.utils.image_dataset_from_directory(
    DATA_DIR,
    image_size=IMG_SIZE_CAT,
    batch_size=BATCH_SIZE,
    label_mode = None,
    seed = 42,
    validation_split = None,
    subset = None,
    shuffle = None,
)

In [ ]:
# Selection d'une partie du dataset

# short_raw_ds = raw_ds.shuffle(buffer_size=1000)
# short_raw_ds = short_raw_ds.take(10)

### 1 Catégorisation des images

Utilisation du modèle construit lors du livrable 1

In [ ]:
# Classification des images
predictions = classification_model.predict(raw_ds, batch_size=BATCH_SIZE)

On garde uniquement les images catégorisé comme étant une photo.

In [ ]:
print(CLASS_NAMES.index("Photo"))

print(f"Nombre d'image avant classification : {len(raw_ds)*BATCH_SIZE} images")

# count the number of value equal to the photo index in the y_pred array
filtered_ds = []
for i, batch in enumerate(raw_ds):
    # Iterate over each image in the batch
    for j in range(batch.shape[0]):
        pred = predictions[i * BATCH_SIZE + j]
        scores = tf.nn.softmax(pred)
        
        # Get the index of the class with the highest probability
        label = np.argmax(scores)
        
        # Check if the label is 'Photo'
        if label == CLASS_NAMES.index("Photo"):
            filtered_ds.append(batch[j])

print(f"Nombre d'image après classification : {len(filtered_ds)} images")


Affichage de 5 images filtrées (tiré aléatoirement).

In [ ]:
import random

start_index = random.randint(0, len(filtered_ds) - 5)

# Display the 5 images in the chosen interval
for i in range(start_index, start_index + 5):
    plt.imshow(filtered_ds[i] / 255)
    plt.axis('off')
    plt.show()

### 2 Débruitage des images

Utilisation du modèle construit lors du livrable 2.

In [ ]:
# Convert filtered_ds (list) to a TensorFlow dataset
filtered_ds_tf = tf.data.Dataset.from_tensor_slices(filtered_ds)

# Rescale images from 180x180 to 256x256
filtered_ds_tf = filtered_ds_tf.map(lambda x: tf.image.resize(x, IMG_SIZE_AUTO_ENCODER))

# Apply the autoencoder to each batch in the dataset
denoised_images = []
for image in filtered_ds_tf:
    denoised_image = autoencoder(tf.expand_dims(image / 255.0, axis=0), training=False)
    denoised_images.append(denoised_image[0])  # Remove batch dimension

Affichages des images débruitées.

In [ ]:
# Concatenate all denoised images into a single tensor
denoised_images = tf.stack(denoised_images, axis=0)

start_index = random.randint(0, len(denoised_images) - 5)

for i in range(start_index, start_index + 5):
    plt.subplot(1, 2, 1)
    plt.imshow(filtered_ds[i] / 255)
    plt.axis('off')
    plt.title("Original Image")
    
    plt.subplot(1, 2, 2)
    plt.imshow(denoised_images[i] / 1)
    plt.axis('off')
    plt.title("Denoised Image")
    
    plt.show()

### 3 Ajout des légendes (captionning)

Utilisation du modèle construit lors du livrable 3.

In [ ]:
for image in denoised_images[:10]: 
    caption = captionning_model.generate_caption(image, batch_size=1)
    plt.imshow(image)
    plt.title(caption)
    plt.show()

Affichages des images légendées.

In [ ]:
# # Concatenate all denoised images into a single tensor
# # caption_images = tf.stack(denoised_images, axis=0)

# start_index = random.randint(0, len(caption_images) - 5)

# for i in range(start_index, start_index + 5):
#     plt.subplot(1, 2, 1)
#     plt.imshow(caption_images[i] / 255)
#     plt.axis('off')
#     plt.title("Final Image")
    
#     plt.show()